# `torch.cuda.Event` 是什么

`torch.cuda.Event` 是 PyTorch 中用于 **异步同步** 和 **耗时测量** 的一个重要工具。

在 CUDA 编程中，GPU 操作通常是异步执行的（即 CPU 发出指令后不等 GPU 跑完就继续往下走了）。`Event` 就像是我们在 GPU 操作流中安插的“路标”或“计时器”，用来记录特定时刻的状态。

我们可以一起探讨它的两个核心用途。

1.  **精准测量耗时 (Time Measurement)** ⏱️
    测量 GPU 上一段操作代码运行的真实时间。普通的 Python `time.time()` 往往测不准 GPU 的实际工作时间，因为它是基于 CPU 的。
2.  **流同步 (Stream Synchronization)** 🚦
    在多个计算流（Streams）之间同步操作，确保某个任务在另一个任务记录下“路标”后再开始，避免数据竞争。


## 精准测量耗时

### 一个简单的“耗时测量”例子

我们可以先看一个测量矩阵乘法耗时的代码片段：

看完这个例子，你觉得代码中的 `torch.cuda.synchronize()` 这一行起到了什么关键作用？如果你把它删掉，测量出来的结果还准确吗？

In [ ]:
import torch

# 准备数据
a = torch.randn(10000, 10000).cuda()
b = torch.randn(10000, 10000).cuda()

# 1. 创建 Event 对象
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

In [ ]:
start_event.record()

# 执行 GPU 密集型操作
c = torch.matmul(a, b)

end_event.record()

# 直接尝试计算时间 (单位是毫秒 ms)
elapsed_time = start_event.elapsed_time(end_event)
print(f"耗时: {elapsed_time} ms")

In [ ]:
start_event.record()

# 执行 GPU 密集型操作
c = torch.matmul(a, b)

end_event.record()

# 等待 GPU 完成所有任务（同步）
torch.cuda.synchronize()

# 此时再计算时间 (单位是毫秒 ms)
elapsed_time = start_event.elapsed_time(end_event)
print(f"耗时: {elapsed_time} ms")

### 为什么必须同步？

在 CUDA 的世界里，CPU 和 GPU 是“各忙各的”。当你调用 `end_event.record()` 时，你只是向 GPU 的指令队列里发了一条命令：“嘿，等你忙完手头这些活儿，记得在这个点儿打个卡。”

* **如果没有 synchronize：** CPU 执行完 `record()` 后会立刻向下执行 Python 代码。此时 GPU 可能还在苦哈哈地算矩阵乘法，根本还没走到“打卡点”。如果你这时候强行问 `elapsed_time`，得到的结果往往是无效的，因此也会报错。
* **有了 synchronize：** CPU 会在这里“原地待命”，直到 GPU 把队列里所有的活（包括那个 `end_event` 的记录操作）全部干完。


## 多流同步


理解 **CUDA Stream（流）**，最形象的办法是把它想象成工厂里的**流水线**。


### 什么是 CUDA Stream？

在默认情况下，你写的 PyTorch 代码都在一条 **默认流（Default Stream）** 上运行。这意味着所有的任务（数据拷贝、矩阵运算、激活函数）都是“串行”的：任务 A 必须跑完，任务 B 才能开始。

**多流（Multi-Stream）** 则是指你在 GPU 上开辟了多条并行的流水线。

* **单流（默认）：** 像是一个厨师，洗菜 -> 切菜 -> 炒菜，必须按顺序来。
* **多流：** 像是后厨团队，有人专门洗菜，有人同时在切另一份菜，有人在炒已经切好的菜。大家都在忙，GPU 的利用率就直接起飞了。


### 为什么要用多流？（核心优势：掩盖延迟）

GPU 运算非常快，但**数据传输**（从内存到显存）相对较慢。多流最经典的应用场景就是：**在计算当前 batch 的同时，预先拷贝下一个 batch 的数据。**

* **流 A：** 负责计算第 10 组数据。
* **流 B：** 负责把第 11 组数据从 CPU 搬到 GPU。

如果没有多流，GPU 在搬运数据时，计算核心就会“闲着”；有了多流，搬运和计算就可以**重叠（Overlap）**。

### 代码怎么写？

在 PyTorch 中，使用 `torch.cuda.Stream()` 来创建和管理流：

In [ ]:
# 创建两个独立的流
s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

# 准备一些数据
input_data = torch.randn(1000, 1000).cuda()

# 使用 'with' 语句切换上下文
with torch.cuda.stream(s1):
    # 这个操作在流 s1 中进行
    res1 = torch.matmul(input_data, input_data)

with torch.cuda.stream(s2):
    # 这个操作在流 s2 中进行，它和 s1 是并行的！
    res2 = input_data * 2

# 注意：流是异步的，如果你现在立刻在 CPU 访问 res1，可能还没算完
# 需要同步
torch.cuda.synchronize()

在多流编程中，最怕的就是 **“计算还没跑完，结果就被取走了”**，这个时候结果是没有意义的会报错。

我们可以把 `Event` 想象成一个 **“接力棒”**。如果流 A 是跑第一棒的，流 B 是跑第二棒的，流 B 必须等到流 A 把接力棒（Event）递过来才能起跑。

#### 错误示范：没有同步的情况

在这个例子中，流 B 可能会在流 A 还没算完矩阵乘法时，就去读取 `data` 的值，导致读到的可能是全零或者是错误的结果。


In [10]:
import torch

# 1. 准备超大规模数据，让 GPU 忙起来
size = 10000
a = torch.randn(size, size, device='cuda')
b = torch.randn(size, size, device='cuda')
res = torch.zeros(size, size, device='cuda')

s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

# 2. 错误示范：不使用 Event 同步
with torch.cuda.stream(s1):
    res = torch.matmul(a, b) # 这是一个很重的任务
    # 它传的是 GPU 时钟周期数 / cycles，大致意思是让当前 CUDA stream
    # 在 GPU 上 busy-wait 这么多个 clock cycles。
    torch.cuda._sleep(100_000_000)

with torch.cuda.stream(s2):
    # s2 不等 s1，直接去拿 res 里的数据
    final_res = res * 2

# 3. 检查结果（不使用 print(final_res)，避免隐式同步）
# 我们检查结果是否全为 0。如果同步了，不应该有 0
zero_count = (final_res == 0).sum()
print(f"有 {zero_count} 个元素是 0，说明计算还没完成流 B 就抢跑了。")

有 624640 个元素是 0，说明计算还没完成流 B 就抢跑了。


/tmp/ipykernel_38883/1587489929.py:5: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  torch.cuda.reset_max_memory_allocated()


#### 正确示范：使用 Event 同步

为了确保安全，我们引入 `torch.cuda.Event` 作为路标。


In [8]:
import torch

# 初始化
data = torch.zeros(5000, 5000).cuda()
matrix_a = torch.randn(5000, 5000).cuda()
matrix_b = torch.randn(5000, 5000).cuda()

stream_a = torch.cuda.Stream()
stream_b = torch.cuda.Stream()

# 1. 创建一个 Event 对象（就像一个路标）
calculation_done = torch.cuda.Event()

# 2. 流 A 执行计算
with torch.cuda.stream(stream_a):
    data = torch.matmul(matrix_a, matrix_b)
    # 在流 A 的任务序列里插入一个“打卡”指令
    calculation_done.record(stream_a)

# 3. 流 B 执行处理
with torch.cuda.stream(stream_b):
    # 关键点：让流 B 等待流 A 的打卡信号
    # 注意：这行代码会告诉 GPU：在看到 calculation_done 之前，不要跑流 B 下面的任务
    # 但它不会阻塞 CPU，CPU 会继续往下执行 Python 代码
    stream_b.wait_event(calculation_done)
    result = data * 2

# 仅仅用于例子的检查
stream_b.synchronize()
ok_gpu = torch.equal(result, data * 2)

print("检查通过" if ok_gpu else "错误：结果不匹配")

检查通过



### 为什么不直接用 `torch.cuda.synchronize()`?

你可能会问：“我直接在两个 `with` 之间加一行 `torch.cuda.synchronize()` 不行吗？”

当然可以运行，但**性能很差**。
* **`torch.cuda.synchronize()`**: 是“大停顿”。它会让 **CPU** 停下来，等待 **GPU 所有流**里的所有活儿都干完。就像教官喊“全体集合”，所有人（包括 CPU 和 GPU 的所有流水线）都得原地站好。
* **`stream.wait_event(event)`**: 是“局部协调”。它只让 **GPU 的流 B** 等待 **GPU 的流 A**。CPU 此时是自由的，可以去处理其他逻辑（比如准备下一组数据、打印日志等）。

**总结一下：**
使用 `Event` 进行流间同步，是在保证**数据正确性**的前提下，实现 **CPU 与 GPU 最大化并行**的核心手段。



### 关键点总结

| 特性 | 说明 |
| :--- | :--- |
| **异步性** | 流里的任务对 CPU 是异步的，CPU 发完指令就走。 |
| **顺序性** | **同一个流**里的任务是严格按顺序执行的。 |
| **并行性** | **不同流**之间的任务可以并行（前提是 GPU 资源够用）。 |
| **安全性** | 如果流 A 的结果是流 B 的输入，必须用 `Event` 进行同步，否则 B 可能会读到错误数据。 |

> **冷知识：** 虽然我们说多流是并行的，但 GPU 的硬件单元（SM）是有限的。如果流 A 把 GPU 塞满了，流 B 就算发出了指令也得排队等待硬件资源。

因此除了计时，`Event` 还可以充当不同流（Stream）之间的**信号灯**。

想象你有两个流：一个负责从内存拷贝数据到 GPU（流 A），另一个负责计算（流 B）。你必须保证数据拷贝完了，计算才能开始。

这种方式比 `torch.cuda.synchronize()` 更高效，因为它只让 GPU 的流 B 等待流 A，而**不会让 CPU 停下来**。


In [ ]:
# 通用的良好示范
stream_a = torch.cuda.Stream()
stream_b = torch.cuda.Stream()
ready_event = torch.cuda.Event()

with torch.cuda.stream(stream_a):
    # 在流 A 中执行数据拷贝
    # ... 拷贝代码 ...
    ready_event.record()  # 记录拷贝完成信号

with torch.cuda.stream(stream_b):
    # 流 B 等待流 A 的信号，但不阻塞 CPU
    stream_b.wait_event(ready_event)
    # 只有当 ready_event 被记录后，这里的计算才会真正开始
    # ... 计算代码 ...